Se  plantean  diferentes  ejercicios  basados  en  modelos  preentrenados  de  embeddings 
contextualizados de palabras. 
 
# Ejercicio 1. Embeddings Contextualizados - ELMo 
A diferencia de los embeddings estáticos, que otorgan un único vector a cada palabra 
independientemente del contexto en el que se encuentra, los embeddings 
contextualizados  generan  representaciones  dinámicas  que  varían  según  el  significado 
de la palabra dentro de una frase concreta. Esta característica permite capturar mejor 
la ambigüedad y polisemia presentes en las palabras del lenguaje natural. 
 
Uno  de  los  primeros  modelos  en  proporcionar  estos  embeddings  contextualizados  de 
palabras fue ELMo. Basado en el empleo de redes neuronales recurrentes 
bidireccionales  (biLSTM),  ELMo  genera  vectores  dinámicos  de  palabras  en  función  del 
contexto en el que aparecen. 
De  este  modo,  ELMo  proporciona  embeddings  contextualizados  a  nivel  de  palabra, 
adaptando  su  representación  según  el  contexto  en  el  que  se  encuentra  dentro  de  un 
determinado  texto.  Por  medio  de  TensorFlow  Hub,  podemos  cargar  este  modelo 
mediante el siguiente código de ejemplo

In [1]:
import numpy as np 
import tensorflow as tf 
import tensorflow_hub as hub 
from sklearn.metrics.pairwise import cosine_similarity 
elmo = hub.load("https://tfhub.dev/google/elmo/3")

2025-06-29 16:48:11.470895: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-29 16:48:11.493302: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-06-29 16:48:11.659154: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-06-29 16:48:11.789085: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1751208491.903781   46026 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1751208491.93

Dada la frase de consulta "Dogs are domestic animals." y el conjunto de frases ["Dogs 
are pets.", "This is a dog.", "They are free today."] se pide lo siguiente: 
- Con ELMo, obtén el vector promedio de la frase de consulta y también el de cada 
frase del conjunto de frases. 
- Con ELMo, obtén el vector máximo de la frase de consulta y también el de cada 
frase del conjunto de frases. 
- Para las anteriores vectorizaciones, calcula la similitud coseno entre la frase de 
consulta y el conjunto de frases. Reflexiona sobre qué frases son consideradas 
más  similares  por  estos  modelos  y  qué  vectores  funcionan  mejor  (media  o 
máximo). 
 
Nota: en este ejercicio no es necesario realizar una etapa previa de tokenización.

In [ ]:
sentence= "Dogs are domestic animals." 
conjunto= ["Dogs are pets.", "This is a dog.", "They are free today."] 

In [52]:
output= elmo.signatures['default'](tf.constant([sentence])) # Esto devuelve un diccionario, nos interesa:

embedding_sentence= output['elmo'].numpy()

print("Tenemos un embedding de tamaño", embedding_sentence.shape)

#Calculamos el vector promedio
sentence_vector_promedio= embedding_sentence[0].mean(axis=0)
conjunto_vectores_promedio =[]
print("Tamaño del vector promedio de la frase:",  sentence_vector_promedio.shape)
for s in conjunto:
    output=elmo.signatures['default'](tf.constant([s]))
    embedding_s= output['elmo'].numpy()
    s_v_mean= embedding_s[0].mean(axis=0)
    conjunto_vectores_promedio .append(s_v_mean)

print(conjunto_vectores_promedio )
output = elmo.signatures['default'](tf.constant([sentence]))
embedding_sentence = output['elmo'].numpy()

print("Tenemos un embedding de tamaño:", embedding_sentence.shape)

# Calculamos el vector promedio de la consulta
sentence_vector_promedio = embedding_sentence[0].mean(axis=0)
print("Tamaño del vector promedio de la frase:", sentence_vector_promedio.shape)

# Vector promedio para cada frase del conjunto
conjunto_vectores_promedio = []

for s in conjunto:
    output = elmo.signatures['default'](tf.constant([s]))
    embedding_s = output['elmo'].numpy()
    s_v_mean = embedding_s[0].mean(axis=0)
    conjunto_vectores_promedio.append(s_v_mean)

# Mostrar resultado
print("\nVectores promedio del conjunto:")
for i, vec in enumerate(conjunto_vectores_promedio):
    print(f"- Frase: {conjunto[i]}")
    print(f"  Vector shape: {vec.shape}")


Tenemos un embedding de tamaño (1, 4, 1024)
Tamaño del vector promedio de la frase: (1024,)
[array([-0.0786633 ,  0.0009354 , -0.15537709, ..., -0.33652297,
        0.01641541,  0.20615375], shape=(1024,), dtype=float32), array([-0.26460934, -0.2382618 ,  0.1407984 , ..., -0.10179784,
        0.13338125,  0.2496654 ], shape=(1024,), dtype=float32), array([-0.3655069 , -0.06678982, -0.17631736, ..., -0.2835953 ,
       -0.44601935,  0.18703815], shape=(1024,), dtype=float32)]
Tenemos un embedding de tamaño: (1, 4, 1024)
Tamaño del vector promedio de la frase: (1024,)

Vectores promedio del conjunto:
- Frase: Dogs are pets.
  Vector shape: (1024,)
- Frase: This is a dog.
  Vector shape: (1024,)
- Frase: They are free today.
  Vector shape: (1024,)


In [42]:
#Calculamos el vector max
sentence_vector_max= embedding_sentence[0].max(axis=0)
conjunto_vectores_max =[]

for s in conjunto:
    output = elmo.signatures['default'](tf.constant([s]))
    embedding_s = output['elmo'].numpy()
    s_v_max= embedding_s[0].max(axis=0)
    conjunto_vectores_max.append(s_v_max)


In [60]:
print(conjunto_vectores_promedio[0])
print(sentence_vector_promedio)

[-0.0786633   0.0009354  -0.15537709 ... -0.33652297  0.01641541
  0.20615375]
[-0.24925394 -0.3763137   0.04285502 ... -0.15762666  0.45162833
  0.05097778]


In [ ]:
# Comparar similitud de coseno con frase y las de consulta
from sklearn.metrics.pairwise import cosine_similarity 

for i, frase in enumerate(conjunto):
    cosine_sim= cosine_similarity([sentence_vector_promedio], [conjunto_vectores_promedio[i]])
    print(f"Similaridad de coseno entre el vector medio de la frase {sentence} y el vector medio de la frase de consulta {frase}: {cosine_sim}")
 #cosine_similarity espera matrices 2D  por eso lo pasamos como []

Similaridad de coseno entre el vector medio de la frase Dogs are domestic animals. y el vector medio de la frase de consulta Dogs are pets.: [[0.8564701]]
Similaridad de coseno entre el vector medio de la frase Dogs are domestic animals. y el vector medio de la frase de consulta This is a dog.: [[0.5757736]]
Similaridad de coseno entre el vector medio de la frase Dogs are domestic animals. y el vector medio de la frase de consulta They are free today.: [[0.4364299]]


In [68]:
# Comparar similitud de coseno con frase y las de consulta
from sklearn.metrics.pairwise import cosine_similarity 

for i, frase in enumerate(conjunto):
    cosine_sim= cosine_similarity([sentence_vector_max], [conjunto_vectores_max[i]])
    print(f"Similaridad de coseno entre el maximo de la frase {sentence} y el maximo de frase de consulta {frase}: {cosine_sim}")
 #cosine_similarity espera matrices 2D  por eso lo pasamos como []

Similaridad de coseno entre el maximo de la frase Dogs are domestic animals. y el maximo de frase de consulta Dogs are pets.: [[0.88454366]]
Similaridad de coseno entre el maximo de la frase Dogs are domestic animals. y el maximo de frase de consulta This is a dog.: [[0.74133706]]
Similaridad de coseno entre el maximo de la frase Dogs are domestic animals. y el maximo de frase de consulta They are free today.: [[0.68283683]]


<!-- # Ejercicio 2. Embeddings Contextualizados – BERT
Uno de los modelos más representativos dentro de esta categoría de embeddings es
BERT. A diferencia de ELMo, BERT utiliza una arquitectura basada en mecanismos de
atención y transformers, lo que le permite considerar el contexto completo en el que se
encuentra una determinada palabra al generar su representación vectorial.
De igual forma, BERT proporciona embeddings contextualizados a nivel de palabra en la
salida de su última capa. Una vez obtenidos los outputs generados por BERT, es posible
acceder a las representaciones vectoriales de las palabras procesadas mediante el
atributo last_hidden_state:

"""
tokens = tokenizer(text, padding=True, truncation=True,
return_tensors="pt")
with torch.no_grad():
 outputs = model(**tokens)
outputs.last_hidden_state.numpy()
"""

Dada nuevamente la frase de consulta "Dogs are domestic animals." y el conjunto de
frases ["Dogs are pets.", "This is a dog.", "They are free today."] se pide lo siguiente:
- Con el modelo y tokenizador de BERT 'bert-base-uncased', obtén el vector
promedio de la frase de consulta y el de cada frase del conjunto de frases.
- Con el modelo y tokenizador de BERT 'bert-base-uncased', obtén el vector
máximo de la frase de consulta y el de cada frase del conjunto de frases.
- Para las anteriores vectorizaciones, calcula la similitud coseno entre la frase de
consulta y cada una del conjunto de frases. Reflexiona sobre qué frases son
consideradas más similares por estos modelos y qué vectores funcionan mejor
(media o máximo).
Nota: https://huggingface.co/google-bert/bert-base-uncased -->

# Ejercicio 2. Embeddings Contextualizados – BERT
Uno de los modelos más representativos dentro de esta categoría de embeddings es
BERT. A diferencia de ELMo, BERT utiliza una arquitectura basada en mecanismos de
atención y transformers, lo que le permite considerar el contexto completo en el que se
encuentra una determinada palabra al generar su representación vectorial.
De igual forma, BERT proporciona embeddings contextualizados a nivel de palabra en la
salida de su última capa. Una vez obtenidos los outputs generados por BERT, es posible
acceder a las representaciones vectoriales de las palabras procesadas mediante el
atributo last_hidden_state:

"""
tokens = tokenizer(text, padding=True, truncation=True,
return_tensors="pt")
with torch.no_grad():
 outputs = model(**tokens)
outputs.last_hidden_state.numpy()
"""

Dada nuevamente la frase de consulta "Dogs are domestic animals." y el conjunto de
frases ["Dogs are pets.", "This is a dog.", "They are free today."] se pide lo siguiente:
- Con el modelo y tokenizador de BERT 'bert-base-uncased', obtén el vector
promedio de la frase de consulta y el de cada frase del conjunto de frases.
- Con el modelo y tokenizador de BERT 'bert-base-uncased', obtén el vector
máximo de la frase de consulta y el de cada frase del conjunto de frases.
- Para las anteriores vectorizaciones, calcula la similitud coseno entre la frase de
consulta y cada una del conjunto de frases. Reflexiona sobre qué frases son
consideradas más similares por estos modelos y qué vectores funcionan mejor
(media o máximo).
Nota: https://huggingface.co/google-bert/bert-base-uncased

In [ ]:
from transformers import BertTokenizer, BertModel
import torch
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Modelo y tokenizador
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')


/home/vc/miniconda3/envs/PLN/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
sentence = "Dogs are domestic animals."
conjunto = ["Dogs are pets.", "This is a dog.", "They are free today."]

tokens = tokenizer(sentence, padding=True, truncation=True,
return_tensors="pt")

with torch.no_grad():
 outputs = model(**tokens)

outputs.last_hidden_state.numpy()



array([[[-0.04590826,  0.23140775, -0.51940924, ..., -0.56938666,
          0.3182961 ,  0.6422323 ],
        [ 0.74140126,  0.66017896, -0.48483372, ..., -0.8740944 ,
          0.83182883,  0.47498176],
        [ 0.39236334,  0.53354347, -0.20232703, ..., -0.8505933 ,
         -0.24867135,  0.62446886],
        ...,
        [ 0.67910254,  0.899607  , -0.16718438, ..., -1.208657  ,
         -0.19453485, -0.32090628],
        [ 0.64640355,  0.01034878, -0.6051321 , ...,  0.12199461,
         -0.4454124 , -0.34644777],
        [ 0.7502387 , -0.02533945, -0.19976042, ..., -0.00711122,
         -0.40182915, -0.14920262]]], shape=(1, 7, 768), dtype=float32)